# Day 047 — Exercise 5: StatsReport

**What you'll build:** The `StatsReport` class — `load(df, columns=None) -> StatsReport` (fluent builder) and `report() -> dict` — run `describe_distribution` + `test_normality` on every numeric column and return the results as a nested dict.

**Why it matters:** When you get a new dataset, the first thing you always do is characterise every column. `StatsReport` automates that: one `.load(df).report()` call gives you the full statistical picture of an entire DataFrame.

## Provided: All Helper Functions

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

def make_sample_data(n: int = 100, seed: int = 42) -> pd.DataFrame:
    """Return a reproducible multi-column dataset for statistics exercises."""
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        'normal_col': rng.standard_normal(n).round(3),
        'skewed_col': rng.exponential(2, n).round(3),
        'score_a':    (50 + rng.standard_normal(n) * 10).round(1),
        'score_b':    (70 + rng.standard_normal(n) * 10).round(1),
    })


def describe_distribution(series: pd.Series) -> dict:
    s   = series.dropna()
    q25 = float(s.quantile(0.25))
    q75 = float(s.quantile(0.75))
    return {
        'count':    int(len(s)),
        'mean':     round(float(s.mean()), 4),
        'median':   round(float(s.median()), 4),
        'std':      round(float(s.std(ddof=1)), 4),
        'sem':      round(float(s.sem()), 4),
        'min':      round(float(s.min()), 4),
        'max':      round(float(s.max()), 4),
        'q25':      round(q25, 4),
        'q75':      round(q75, 4),
        'iqr':      round(q75 - q25, 4),
        'skewness': round(float(s.skew()), 4),
        'kurtosis': round(float(s.kurt()), 4),
    }


def test_normality(series: pd.Series, alpha: float = 0.05) -> dict:
    s      = series.dropna()
    stat, p = stats.shapiro(s)
    return {
        'n':          len(s),
        'statistic':  round(float(stat), 4),
        'p_value':    round(float(p), 6),
        'is_normal':  bool(p > alpha),
        'alpha':      alpha,
    }


def correlation_with_pvalue(x: pd.Series, y: pd.Series,
                             method: str = 'pearson') -> dict:
    mask  = x.notna() & y.notna()
    x_c, y_c = x[mask], y[mask]
    if method == 'pearson':
        r, p = stats.pearsonr(x_c, y_c)
    elif method == 'spearman':
        r, p = stats.spearmanr(x_c, y_c)
    else:
        raise ValueError(f"method must be 'pearson' or 'spearman', got {method!r}")
    return {
        'method':         method,
        'n':              len(x_c),
        'r':              round(float(r), 4),
        'p_value':        round(float(p), 6),
        'is_significant': bool(p < 0.05),
    }


def compare_groups(a: pd.Series, b: pd.Series,
                   alpha: float = 0.05) -> dict:
    a_c, b_c  = a.dropna(), b.dropna()
    t, p       = stats.ttest_ind(a_c, b_c)
    n_a, n_b   = len(a_c), len(b_c)
    std_a      = float(a_c.std(ddof=1))
    std_b      = float(b_c.std(ddof=1))
    pooled_var = ((n_a - 1) * std_a**2 + (n_b - 1) * std_b**2) / (n_a + n_b - 2)
    pooled     = np.sqrt(pooled_var) if pooled_var > 0 else 0.0
    d          = (float(a_c.mean()) - float(b_c.mean())) / pooled if pooled > 0 else 0.0
    sig        = bool(p < alpha)
    return {
        'n_a':            n_a,
        'n_b':            n_b,
        'mean_a':         round(float(a_c.mean()), 4),
        'mean_b':         round(float(b_c.mean()), 4),
        'statistic':      round(float(t), 4),
        'p_value':        round(float(p), 6),
        'is_significant': sig,
        'cohens_d':       round(d, 4),
        'conclusion':     'different' if sig else 'not_different',
    }

## Your Implementation

In [ ]:
class StatsReport:
    """
    Automated per-column statistical report.

    Usage:
        sr = StatsReport()
        report = sr.load(df).report()
    """

    def __init__(self):
        # TODO: self._df      = None
        # TODO: self._columns = None
        pass

    def load(self, df: pd.DataFrame,
             columns: list | None = None) -> 'StatsReport':
        """
        Store df and the columns to analyse.
        If columns is None, use all numeric columns.
        Returns self for fluent chaining.
        """
        # TODO: self._df      = df.copy()
        # TODO: num_cols      = df.select_dtypes(include='number').columns.tolist()
        # TODO: self._columns = columns if columns is not None else num_cols
        # TODO: return self
        pass

    def report(self) -> dict:
        """
        Return {col: {'distribution': {...}, 'normality': {...}}} for each column.
        Skip columns not in df or with fewer than 3 non-null values.
        """
        # TODO: result = {}
        # TODO: for col in self._columns:
        #     if col not in self._df.columns: continue
        #     s = self._df[col].dropna()
        #     if len(s) < 3: continue
        #     result[col] = {
        #         'distribution': describe_distribution(s),
        #         'normality':    test_normality(s),
        #     }
        # TODO: return result
        pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    df = make_sample_data(100)

    # Check 1: class defined with load and report methods
    try:
        assert 'StatsReport' in globals()
        for m in ('load', 'report'):
            assert hasattr(StatsReport, m), f'missing method: {m}'
        passed += 1; print('\u2705 Check 1: StatsReport has load and report')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: load() returns self (fluent)
    try:
        sr  = StatsReport()
        ret = sr.load(df)
        assert ret is sr, f'load() must return self, got {type(ret).__name__}'
        passed += 1; print('\u2705 Check 2: load() returns self (fluent chaining)')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: report() returns non-empty dict
    try:
        rep = sr.report()
        assert isinstance(rep, dict), \
            f'report() should return dict, got {type(rep).__name__}'
        assert len(rep) > 0, 'report() returned empty dict'
        passed += 1; print(f'\u2705 Check 3: report() returns dict with {len(rep)} entries')
    except Exception as e:
        print(f'\u274c Check 3: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 4: report keys are column names from the DataFrame
    try:
        for col in rep:
            assert col in df.columns, f'unexpected column in report: {col!r}'
        assert len(rep) == 4, f'expected 4 numeric columns, got {len(rep)}'
        passed += 1; print(f'\u2705 Check 4: 4 numeric columns reported')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: each entry has distribution and normality sub-dicts
    try:
        col = list(rep.keys())[0]
        entry = rep[col]
        assert 'distribution' in entry, f'missing distribution key'
        assert 'normality'    in entry, f'missing normality key'
        dist = entry['distribution']
        norm = entry['normality']
        assert 'count'    in dist, 'distribution missing count'
        assert 'skewness' in dist, 'distribution missing skewness'
        assert 'is_normal' in norm, 'normality missing is_normal'
        passed += 1; print(f'\u2705 Check 5: each entry has distribution + normality sub-dicts')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
class StatsReport:
    def __init__(self):
        self._df      = None
        self._columns = None

    def load(self, df: pd.DataFrame,
             columns: list | None = None) -> 'StatsReport':
        self._df      = df.copy()
        num_cols      = df.select_dtypes(include='number').columns.tolist()
        self._columns = columns if columns is not None else num_cols
        return self

    def report(self) -> dict:
        result = {}
        for col in self._columns:
            if col not in self._df.columns:
                continue
            s = self._df[col].dropna()
            if len(s) < 3:
                continue
            result[col] = {
                'distribution': describe_distribution(s),
                'normality':    test_normality(s),
            }
        return result
```

</details>